# Fraud Detection on BankSim Transactions

## Project Overview

This project develops a machine learning system to detect fraudulent transactions using the **BankSim synthetic banking dataset**.

The system analyzes customer transaction behavior and identifies anomalies that may indicate fraudulent activity.

Unlike traditional fraud detection systems that only output a binary prediction, this project introduces a **dual scoring framework**:

- **Fraud Likelihood** – probability that a transaction is fraudulent
- **Detection Confidence** – reliability of the fraud prediction based on behavioral consistency

This approach helps distinguish between isolated anomalies and persistent suspicious behavior.

# Dataset Description

The dataset used in this project is **BankSim**, a synthetic dataset generated by an agent-based simulator of bank payments.

The simulator models realistic banking behavior and injects fraudulent transactions to simulate real-world fraud scenarios.

### Key Dataset Characteristics

- ~594,000 transactions
- ~7,200 fraudulent transactions
- ~1.2% fraud rate
- Transactions simulated over 6 months

### Key Features

| Feature | Description |
|------|------|
| step | Time step of transaction |
| customer | Customer ID |
| age | Age group |
| gender | Customer gender |
| zipcodeOri | Customer location |
| merchant | Merchant ID |
| zipMerchant | Merchant location |
| category | Transaction category |
| amount | Transaction amount |
| fraud | Fraud label (0 = normal, 1 = fraud) |

The dataset is **synthetic but behaviorally realistic**, making it suitable for fraud detection research without privacy concerns.

## Additional Dataset

The second dataset used in this project is the **IEEE-CIS Fraud Detection Dataset**, a real-world anonymized dataset released as part of a Kaggle competition for financial fraud detection.

The dataset was provided by **Vesta Corporation and IEEE**, containing online transaction records along with device and identity information. It is widely used in fraud detection research because it combines **transaction behavior with user identity signals**, enabling more advanced fraud detection models.

### Key Dataset Characteristics

- ~590,000 transactions
- Highly imbalanced fraud distribution
- Over 430 anonymized features
- Includes both transaction data and identity/device information

### Key Files Used

| File | Description |
|------|------|
| train_transaction.csv | Transaction-level data including payment information and fraud labels |
| train_identity.csv | Device, browser, and identity-related attributes associated with transactions |

These two files are linked using the **TransactionID** field and are merged to create a unified dataset for analysis.

### Key Features

| Feature | Description |
|------|------|
| TransactionID | Unique identifier for each transaction |
| TransactionDT | Time delta from a reference point |
| TransactionAmt | Transaction payment amount |
| ProductCD | Product code of the transaction |
| card1–card6 | Encoded card-related attributes |
| addr1, addr2 | Billing region information |
| DeviceType | Type of device used for the transaction |
| DeviceInfo | Device or browser information |
| id_12 – id_38 | Anonymized identity and network features |
| isFraud | Fraud label (0 = normal, 1 = fraud) |

The dataset contains a mixture of **transactional, behavioral, and device-level attributes**, allowing models to detect fraud patterns such as unusual spending behavior, device changes, or location inconsistencies.

The features are partially anonymized to protect sensitive financial information while still preserving realistic fraud patterns for machine learning research.


# Import Libraries

This section imports the necessary Python libraries used throughout the project.

Key libraries include:

- **pandas / numpy** for data manipulation
- **matplotlib / seaborn** for visualization
- **scikit-learn** for machine learning models

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Machine learning models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Optional: handling class imbalance
from sklearn.utils import resample

# Visualization style
sns.set(style="whitegrid")

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Exploratory Data Analysis (EDA)

Exploratory Data Analysis is performed to better understand the dataset and identify potential issues.

Key analysis steps include:

- Checking dataset structure
- Inspecting data types
- Analyzing statistical distributions
- Observing fraud class imbalance

Understanding these characteristics helps guide feature engineering and model selection.

## EDA for BankSim Dataset

In [ ]:
# Load Dataset

# Unload the dataset from the zip file
import zipfile

with zipfile.ZipFile("../bs140513_032310.zip", 'r') as zip_ref:
    zip_ref.extractall("../data")


#Load dataset
df = pd.read_csv("../data/bs140513_032310.csv")

In [ ]:
# Basic dataset overview
print("Dataset shape:", df.shape)
print("\nDataset info:")
df.info()

print("\nSummary statistics:")
display(df.describe())

# Check missing values
print("\nMissing values per column:")
print(df.isnull().sum())

In [ ]:
# Fraud distribution
plt.figure(figsize=(6,4))
sns.countplot(x='fraud', data=df)
plt.title("Fraud vs Non-Fraud Transactions")
plt.xlabel("Fraud")
plt.ylabel("Count")
plt.show()

# Percentage distribution
fraud_percent = df['fraud'].value_counts(normalize=True) * 100
print("Fraud percentage distribution:")
print(fraud_percent)

In [ ]:
# Transaction amount distribution
plt.figure(figsize=(8,5))
sns.histplot(df['amount'], bins=50, kde=True)
plt.title("Transaction Amount Distribution")
plt.xlabel("Amount")
plt.ylabel("Frequency")
plt.show()

### Transaction Amount Distribution

The histogram shows that transaction amounts are highly right-skewed. 
Most transactions involve relatively small amounts, while a small number 
of transactions have significantly larger values.

This pattern is common in financial transaction datasets where everyday 
purchases are small but occasional large payments occur.

To reduce skewness and improve model performance, a logarithmic 
transformation is applied to the transaction amount feature.

In [ ]:
df['amount_log'] = np.log1p(df['amount'])

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['amount_log'], bins=50, kde=True)
plt.title("Log-Transformed Transaction Amount Distribution")
plt.xlabel("Log(Amount)")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Transaction amount by fraud status
plt.figure(figsize=(8,5))
sns.boxplot(x='fraud', y='amount', data=df)
plt.title("Transaction Amount by Fraud Status")
plt.xlabel("Fraud")
plt.ylabel("Amount")
plt.show()

In [ ]:
# Transaction categories distribution
plt.figure(figsize=(10,5))
sns.countplot(y='category', data=df, order=df['category'].value_counts().index)
plt.title("Transaction Categories")
plt.xlabel("Count")
plt.ylabel("Category")
plt.show()

In [ ]:
# Fraud rate by category
fraud_by_category = df.groupby('category')['fraud'].mean().sort_values(ascending=False)

plt.figure(figsize=(10,5))
fraud_by_category.plot(kind='bar')
plt.title("Fraud Rate by Transaction Category")
plt.ylabel("Fraud Rate")
plt.xlabel("Category")
plt.show()

In [ ]:
# Fraud rate by age group
fraud_by_age = df.groupby('age')['fraud'].mean()

plt.figure(figsize=(8,5))
fraud_by_age.plot(kind='bar')
plt.title("Fraud Rate by Age Group")
plt.ylabel("Fraud Rate")
plt.xlabel("Age Group")
plt.show()

In [ ]:
# Fraud rate by gender
fraud_by_gender = df.groupby('gender')['fraud'].mean()

plt.figure(figsize=(6,4))
fraud_by_gender.plot(kind='bar')
plt.title("Fraud Rate by Gender")
plt.ylabel("Fraud Rate")
plt.xlabel("Gender")
plt.show()

In [ ]:
# Top merchants by transaction count
top_merchants = df['merchant'].value_counts().head(10)

plt.figure(figsize=(8,5))
sns.barplot(x=top_merchants.values, y=top_merchants.index)
plt.title("Top 10 Merchants by Transaction Count")
plt.xlabel("Transactions")
plt.ylabel("Merchant")
plt.show()

In [ ]:
# Transaction frequency over time (by step)
plt.figure(figsize=(10,5))
sns.histplot(df['step'], bins=50)
plt.title("Transaction Frequency Over Time")
plt.xlabel("Step")
plt.ylabel("Transactions")
plt.show()

In [ ]:
# Correlation heatmap for numeric features
numeric_cols = df.select_dtypes(include=['int64','float64'])

plt.figure(figsize=(8,6))
sns.heatmap(numeric_cols.corr(), annot=True, cmap='coolwarm')
plt.title("Feature Correlation Heatmap")
plt.show()

## EDA for IEEE-CIS dataset

### Load and Merge the Dataset

The IEEE-CIS dataset is provided as two separate files:

train_transaction.csv

train_identity.csv

These files are linked using TransactionID.

In [ ]:
# Load the IEEE-CIS Fraud Detection dataset
train_transaction = pd.read_csv("train_transaction.csv")
train_identity = pd.read_csv("train_identity.csv")

df_ieee = train_transaction.merge(train_identity, on="TransactionID", how="left")

print("Dataset Shape:", df_ieee.shape)
df_ieee.head()

In [ ]:
# Basic dataset overview
df_ieee.info()
df_ieee.describe()


In [ ]:
# Check missing values
missing_values = df_ieee.isnull().sum().sort_values(ascending=False)

missing_values.head(20)

In [ ]:
# Visualize missing values
plt.figure(figsize=(10,6))
missing_values[:20].plot(kind='bar')
plt.title("Top Features with Missing Values")
plt.ylabel("Missing Count")
plt.show()

In [ ]:
# Fraud distribution
fraud_counts = df_ieee['isFraud'].value_counts()

print(fraud_counts)

In [ ]:
# Visualize fraud distribution
sns.countplot(x='isFraud', data=df_ieee)
plt.title("Fraud vs Non-Fraud Distribution")

plt.show()

In [ ]:
# Transaction amount distribution
plt.figure(figsize=(10,6))
sns.histplot(df_ieee['TransactionAmt'], bins=50)
plt.title("Transaction Amount Distribution")
plt.xlabel("Transaction Amount")
plt.show()

In [ ]:
# Transaction amount by fraud class
plt.figure(figsize=(10,6))
sns.boxplot(x='isFraud', y='TransactionAmt', data=df_ieee)
plt.title("Transaction Amount by Fraud Class")
plt.show()

In [ ]:
# Transaction distribution by device type
sns.countplot(x='DeviceType', data=df_ieee)
plt.title("Transaction Distribution by Device Type")
plt.show()

In [ ]:
# Fraud distribution by device type
sns.countplot(x='DeviceType', hue='isFraud', data=df_ieee)
plt.title("Fraud Distribution by Device Type")
plt.show()

In [ ]:
# Transaction distribution by product code
sns.countplot(x='ProductCD', data=df_ieee)
plt.title("Transaction Distribution by Product Code")
plt.show()

In [ ]:
# Fraud distribution by product code
sns.countplot(x='ProductCD', hue='isFraud', data=df_ieee)
plt.title("Fraud Distribution by Product Code")
plt.show()

In [ ]:
# Correlation heatmap for numeric features
numeric_cols = df_ieee.select_dtypes(include=['float64','int64']).columns[:20]

plt.figure(figsize=(12,8))
sns.heatmap(df_ieee[numeric_cols].corr(), cmap="coolwarm")
plt.title("Feature Correlation Heatmap")
plt.show()

Key findings from the exploratory data analysis include:

- The dataset is highly imbalanced, with fraudulent transactions forming a small proportion.
- Transaction amount shows wide variability and potential outliers.
- Device and identity features introduce additional behavioral signals useful for fraud detection.
- A significant number of features contain missing values and require preprocessing.
- Some categorical variables such as product code and device type show different distributions between fraud and non-fraud transactions.

# Data Cleaning & Feature Engineering

Before training machine learning models, the dataset must be cleaned.

Cleaning steps include:

- Removing duplicate transactions
- Handling missing values
- Filtering invalid transaction values
- Ensuring correct data types

These steps ensure that the dataset is reliable and suitable for model training.

Feature engineering transforms raw transaction data into meaningful variables that help machine learning models detect fraud patterns.

This project focuses on **behavior-based feature engineering**, which captures deviations from normal customer behavior.

Key feature types include:

- Transaction amount transformations
- Customer spending statistics
- Transaction frequency indicators
- Merchant and category novelty detection
- Sequential anomaly tracking

These features help identify suspicious patterns that may indicate fraudulent activity.

## Data Cleaning and Feature Engineering for BankSim Dataset

In [ ]:
# 1. Remove duplicate records
print("Number of duplicate rows:", df.duplicated().sum())

df = df.drop_duplicates()

print("Shape after removing duplicates:", df.shape)

In [ ]:
# 2. Check missing values
print("\nMissing values per column:")
print(df.isnull().sum())

In [ ]:
#3. Remove transactions with negative amounts
df = df[df['amount'] >= 0]
print("Shape after removing negative amounts:", df.shape)


In [ ]:
#4. Convert Data Types

# Convert age to string
df['age'] = df['age'].astype(str)

# Remove quotes
df['age'] = df['age'].str.replace("'", "")

# Replace unknown values
df['age'] = df['age'].replace('U', 'unknown')

# One-hot encode age groups
df = pd.get_dummies(df, columns=['age'], prefix='age')

# Convert fraud column
df['fraud'] = df['fraud'].astype(int)

# Convert step column
df['step'] = df['step'].astype(int)

In [ ]:
#4. Convert Data Types
df = pd.get_dummies(df, columns=['gender', 'category'], drop_first=True)

In [ ]:
#5. Clean String Columns
# Remove unwanted quotes from string columns
df['customer'] = df['customer'].str.replace("'", "")
df['merchant'] = df['merchant'].str.replace("'", "")


In [ ]:
#6. Verify Clean Dataset
# Check dataset after cleaning
df.info()

# Check for remaining missing values
df.isnull().sum()

# View cleaned dataset
df.head()

## Data Cleaning and Feature Engineering for IEEE-CIS Dataset

In [ ]:
missing_ratio = df_ieee.isnull().mean().sort_values(ascending=False)
missing_ratio.head(20)

In [ ]:
threshold = 0.9
cols_to_drop = missing_ratio[missing_ratio > threshold].index

df_ieee_clean = df_ieee.drop(columns=cols_to_drop)

print("Remaining columns:", df_ieee_clean.shape[1])

In [ ]:
# numeric columns
num_cols = df_ieee_clean.select_dtypes(include=['float64','int64']).columns

df_ieee_clean[num_cols] = df_ieee_clean[num_cols].fillna(
    df_ieee_clean[num_cols].median()
)

# categorical columns
cat_cols = df_ieee_clean.select_dtypes(include=['object']).columns

df_ieee_clean[cat_cols] = df_ieee_clean[cat_cols].fillna("Unknown")

In [ ]:
sparse_cols = df_ieee_clean.columns[df_ieee_clean.nunique() <= 1]
df_ieee_clean = df_ieee_clean.drop(columns=sparse_cols)

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

for col in cat_cols:
    df_ieee_clean[col] = label_encoder.fit_transform(df_ieee_clean[col])

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

df_ieee_clean[num_cols] = scaler.fit_transform(df_ieee_clean[num_cols])

In [ ]:
import numpy as np

df_ieee_clean["TransactionAmt_log"] = np.log1p(df_ieee["TransactionAmt"])

In [ ]:
df_ieee_clean["Transaction_hour"] = (df_ieee["TransactionDT"] // 3600) % 24
df_ieee_clean["Transaction_day"] = (df_ieee["TransactionDT"] // (3600*24)) % 7

In [ ]:
mean_amt = df_ieee["TransactionAmt"].mean()

df_ieee_clean["amount_deviation"] = df_ieee["TransactionAmt"] - mean_amt

In [ ]:
df_ieee_clean["device_missing"] = df_ieee["DeviceInfo"].isnull().astype(int)

In [ ]:
selected_features = [
    "TransactionAmt",
    "TransactionAmt_log",
    "Transaction_hour",
    "Transaction_day",
    "amount_deviation",
    "ProductCD",
    "DeviceType",
    "card1",
    "card2",
    "addr1",
]

X = df_ieee_clean[selected_features]
y = df_ieee_clean["isFraud"]

In [ ]:
print("Final Feature Set Shape:", X.shape)
print("Target Distribution:")
print(y.value_counts())

# Behavioral Baseline Modeling

Fraud detection often relies on identifying deviations from a customer's normal transaction behavior.

To capture this behavior, rolling statistical features are computed for each customer, including:

- Rolling average transaction amount
- Rolling standard deviation of spending

These statistics represent a **behavioral baseline** that defines typical spending patterns.

Transactions that deviate significantly from this baseline may indicate suspicious activity.

In [ ]:
df = df.sort_values(['customer', 'step'])

df['transaction_count'] = df.groupby('customer').cumcount()

In [ ]:
df['customer_avg_amount'] = df.groupby('customer')['amount'].transform('mean')

# Anomaly Detection Features

Anomaly features measure how unusual a transaction is compared to historical behavior.

Examples include:

- Transaction amount z-score
- New merchant detection
- New category detection
- Transaction burst detection
- High deviation flags

These signals help the model identify transactions that differ from typical behavior patterns.

In [ ]:
df['amount_deviation'] = df['amount'] - df['customer_avg_amount']

In [ ]:
df['rolling_mean_5'] = (
    df.groupby('customer')['amount']
    .transform(lambda x: x.shift(1).rolling(5).mean())
)

In [ ]:
df['rolling_std_5'] = (
    df.groupby('customer')['amount']
    .transform(lambda x: x.shift(1).rolling(5).std())
)

In [ ]:
df['amount_zscore'] = (
    (df['amount'] - df['rolling_mean_5']) / df['rolling_std_5']
)

In [ ]:
df['new_merchant'] = (
    df.groupby('customer')['merchant']
    .transform(lambda x: ~x.duplicated())
).astype(int)

In [ ]:
df['merchant_freq'] = df.groupby(['customer','merchant']).cumcount()

In [ ]:
df['recent_txn_count'] = (
    df.groupby('customer')['step']
    .transform(lambda x: x.shift(1).rolling(10).count())
)

In [ ]:
df['high_amount_flag'] = (df['amount'] > df['amount'].quantile(0.95)).astype(int)

In [ ]:
df = df.fillna(0)

In [ ]:
df_model = df.drop(columns=['customer','merchant','zipcodeOri','zipMerchant'])

In [ ]:
df_model.info()

# Data Preparation for Machine Learning

Before training models, the dataset must be converted into a format suitable for machine learning algorithms.

Key preparation steps include:

- Encoding categorical variables
- Scaling numerical features
- Splitting the dataset into training and testing sets

The training set is used to train the model, while the testing set evaluates model performance on unseen data.

## Data Prep for BankSim Dataset

In [ ]:
X = df_model.drop(columns=['fraud'])
y = df_model['fraud']

In [ ]:
print(X.shape)
print(y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=False
)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Data Prep for IEEE-CIS Dataset

In [ ]:
# target variable
y = df_ieee_clean["isFraud"]

# feature variables
X = df_ieee_clean.drop(columns=["isFraud", "TransactionID"])

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts())
print("After SMOTE:", y_train_resampled.value_counts())

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_resampled = scaler.fit_transform(X_train_resampled)
X_test = scaler.transform(X_test)

# Fraud Detection Model

Machine learning models are trained to classify transactions as fraudulent or legitimate.

Common models used in fraud detection include:

- Logistic Regression
- Random Forest
- Gradient Boosting

These models learn patterns in the transaction data and predict the probability that a transaction is fraudulent.

In [ ]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(
    class_weight='balanced',
    max_iter=5000
)

log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report

print("Logistic Regression Evaluation:\n")
print(classification_report(y_test, y_pred_log))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred_log)

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')

plt.title("Logistic Regression Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred_log)
print("Accuracy:", accuracy)

In [ ]:
fraud_prob_log = log_model.predict_proba(X_test)[:,1]

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

fpr, tpr, thresholds = roc_curve(y_test, fraud_prob_log)
auc_score = roc_auc_score(y_test, fraud_prob_log)

plt.figure(figsize=(6,4))
plt.plot(fpr, tpr, label=f"AUC = {auc_score:.3f}")
plt.plot([0,1],[0,1],'--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Logistic Regression")
plt.legend()

plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, _ = precision_recall_curve(y_test, fraud_prob_log)

plt.figure(figsize=(6,4))
plt.plot(recall, precision)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")

plt.show()

In [ ]:
import pandas as pd

feature_importance = pd.Series(
    log_model.coef_[0],
    index=X.columns
).sort_values(ascending=False)

print(feature_importance.head(30))

In [ ]:
feature_importance.head(10).plot(kind='barh')
plt.title("Top Logistic Regression Features")
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report

print("Random Forest Evaluation:\n")
print(classification_report(y_test, y_pred_rf))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred_rf)

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')

plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred_rf)
print("Accuracy:", accuracy)

In [ ]:
fraud_prob_rf = rf_model.predict_proba(X_test)[:,1]

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

fpr, tpr, thresholds = roc_curve(y_test, fraud_prob_rf)
auc_score = roc_auc_score(y_test, fraud_prob_rf)

plt.figure(figsize=(6,4))
plt.plot(fpr, tpr, label=f"AUC = {auc_score:.3f}")
plt.plot([0,1],[0,1],'--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Random Forest")
plt.legend()

plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, _ = precision_recall_curve(y_test, fraud_prob_rf)

plt.figure(figsize=(6,4))
plt.plot(recall, precision)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")

plt.show()

In [ ]:
import pandas as pd

rf_feature_importance = pd.Series(
    rf_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print(rf_feature_importance.head(10))

In [ ]:
rf_feature_importance.head(10).plot(kind='barh')
plt.title("Top Fraud Detection Features")
plt.xlabel("Feature Importance")
plt.ylabel("Features")
plt.show()

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb_model = GradientBoostingClassifier()

gb_model.fit(X_train, y_train)

y_pred_gb = gb_model.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report

print("Gradient Boosting Evaluation:\n")
print(classification_report(y_test, y_pred_gb))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred_gb)

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')

plt.title("Gradient Boosting Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred_gb)
print("Accuracy:", accuracy)

In [ ]:
fraud_prob_gb = gb_model.predict_proba(X_test)[:,1]

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

fpr, tpr, thresholds = roc_curve(y_test, fraud_prob_gb)
auc_score = roc_auc_score(y_test, fraud_prob_gb)

plt.figure(figsize=(6,4))
plt.plot(fpr, tpr, label=f"AUC = {auc_score:.3f}")
plt.plot([0,1],[0,1],'--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Logistic Regression")
plt.legend()

plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, _ = precision_recall_curve(y_test, fraud_prob_gb)

plt.figure(figsize=(6,4))
plt.plot(recall, precision)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")

plt.show()

In [ ]:
import pandas as pd

gb_feature_importance = pd.Series(
    gb_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print(gb_feature_importance.head(10))

In [ ]:
gb_feature_importance.head(10).plot(kind='barh')
plt.title("Top Fraud Detection Features")
plt.xlabel("Feature Importance")
plt.ylabel("Features")
plt.show()

## Confidence Model

In addition to predicting the probability that a transaction is fraudulent, the system also estimates a **detection confidence score**. 

While the fraud probability indicates how suspicious a transaction appears, the confidence score reflects **how reliable that prediction is based on behavioral consistency and anomaly signals**.

This approach separates two important concepts:

- **Fraud likelihood**: the probability that a transaction is fraudulent
- **Detection confidence**: the reliability of that prediction

This distinction is important in fraud detection systems because a single anomaly may not provide enough evidence for strong conclusions. Instead, confidence increases when multiple anomaly indicators appear together.

### Confidence Score Calculation

The confidence score is computed based on several anomaly signals extracted during feature engineering:

- Transaction amount deviation
- Device anomaly
- Location anomaly
- Transaction time anomaly

Each signal contributes to the final confidence score.

The confidence score is calculated as:

Confidence Score = Weighted Sum of Anomaly Indicators

The score is normalized between **0 and 1**, where:

- **Low confidence (near 0)** indicates weak or isolated anomaly signals
- **High confidence (near 1)** indicates strong or repeated anomaly patterns

In [ ]:
# amount anomaly (large deviation from average)
df_ieee_clean["amount_anomaly"] = (
    abs(df_ieee_clean["TransactionAmt"] - df_ieee_clean["TransactionAmt"].mean())
    > df_ieee_clean["TransactionAmt"].std()
).astype(int)


# device anomaly (missing device info)
df_ieee_clean["device_anomaly"] = df_ieee["DeviceInfo"].isnull().astype(int)


# time anomaly (transactions at unusual hours)
df_ieee_clean["time_anomaly"] = (
    (df_ieee_clean["Transaction_hour"] < 5) | (df_ieee_clean["Transaction_hour"] > 23)
).astype(int)

In [ ]:
# weights for each anomaly signal
w_amount = 0.4
w_device = 0.3
w_time = 0.3

df_ieee_clean["confidence_score"] = (
    w_amount * df_ieee_clean["amount_anomaly"]
    + w_device * df_ieee_clean["device_anomaly"]
    + w_time * df_ieee_clean["time_anomaly"]
)

In [ ]:
#normalize confidence score to 0-1 range
df_ieee_clean["confidence_score"] = df_ieee_clean["confidence_score"].clip(0, 1)

In [ ]:
# Fraud Probability using Trained Model
fraud_probability = model.predict_proba(X_test)[:,1]

In [ ]:
# Final Risk Score combining fraud probability and confidence score
final_risk_score = fraud_probability * df_ieee_clean.loc[X_test.index, "confidence_score"]

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.histplot(df_ieee_clean["confidence_score"], bins=20)
plt.title("Confidence Score Distribution")
plt.xlabel("Confidence Score")
plt.show()

The confidence score provides additional insight into fraud detection decisions.

Example interpretations:

| Fraud Probability | Confidence | Interpretation |
|---|---|---|
| High | High | Strong fraud signal |
| High | Low | Suspicious but uncertain |
| Low | High | Stable normal behavior |
| Low | Low | Weak signals overall |

This dual-score framework improves interpretability and helps analysts understand the reliability of model predictions.

# Conclusion

This project demonstrates how machine learning techniques can be applied to detect fraudulent banking transactions using behavioral analysis.

By combining anomaly detection with machine learning classification and confidence scoring, the system provides a more informative risk assessment framework.

Future improvements could include:

- Graph-based fraud detection
- Deep learning models
- Real-time fraud monitoring systems